# 👁 Computer Vision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharifiZarchi/IntroAI/blob/main/Session_08/ComputerVision/Computer_Vision_Tutorial_EN.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FSharifiZarchi%2FIntroAI%2Fblob%2Fmain%2FSession_08%2FComputerVision%2FComputer_Vision_Tutorial_EN.ipynb)

This session covers the operation and the architecture behind modern computer vision: **convolution** and the **convolutional neural network (CNN)**, from a numpy implementation to production-grade pretrained models for classification, detection, and segmentation. The plan:

1. The convolution operation, implemented from scratch
2. From kernels to layers: the CNN building blocks
3. A CNN on CIFAR-10, evaluated against Session 7's fully-connected network
4. Inside the trained CNN: kernels and feature maps
5. Data augmentation with albumentations
6. Transfer learning with ResNet18
7. Professional tools: object detection and semantic segmentation
8. Summary and next steps

Prerequisites: Session 7 (the PyTorch training loop, the fully-connected CIFAR-10 result). **On Colab** everything is preinstalled; the GPU (*Runtime ▸ Change runtime type ▸ T4 GPU*) speeds up Parts 3 and 6 several-fold, though CPU works throughout (the slow cells take about 2 minutes each, and the fine-tuning cell about ten minutes on CPU against about one on the GPU).

**How to run:** click a cell and press `Shift + Enter`, in order, top to bottom.

---
# Part 1: The convolution operation

A **convolution** slides a small grid of weights, the **kernel** (or filter), across an image. At each position the output is the weighted sum of the image patch under the kernel; for a 3×3 kernel:

$$\text{out}(i, j) = \sum_{a=1}^{3} \sum_{b=1}^{3} \; \text{kernel}(a, b) \cdot \text{image}(i+a, j+b)$$

This is the same weighted sum every model of this course computes, with two deliberate restrictions:

- **Locality.** Each output value depends only on a small neighborhood, matching the structure of images: nearby pixels are related, distant ones mostly are not.
- **Weight sharing.** The *same* few weights are applied at every position, so whatever pattern the kernel detects, it detects everywhere in the image. A fully-connected layer, by contrast, ties every weight to one absolute pixel position.

The implementation is a direct transcription of the formula:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def convolve(image, kernel):
    kh, kw = kernel.shape
    H, W = image.shape
    out = np.zeros((H - kh + 1, W - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i:i+kh, j:j+kw] * kernel).sum()
    return out

print("convolution implemented ✅")

What a kernel does is best seen on a real photograph (a few openly licensed sample photos live in the `images` folder beside this notebook; any missing file is fetched automatically from the course repository). The kernel below computes left column minus right column of every 3×3 patch, so it responds exactly where brightness changes horizontally, which is a **vertical edge**:

In [ ]:
import os
from urllib.request import urlretrieve

BASE_URL = "https://raw.githubusercontent.com/SharifiZarchi/IntroAI/main/Session_08/ComputerVision/images/"

def load_image(name):
    if not os.path.exists(os.path.join("images", name)):
        os.makedirs("images", exist_ok=True)
        urlretrieve(BASE_URL + name, os.path.join("images", name))
    return plt.imread(os.path.join("images", name))     # float array in [0, 1]

image = load_image("camera.png")
print("image shape:", image.shape)

vertical_edge = np.array([[1.0, 0.0, -1.0],
                          [1.0, 0.0, -1.0],
                          [1.0, 0.0, -1.0]])

feature_map = convolve(image, vertical_edge)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
ax1.imshow(image, cmap="gray"); ax1.set_title("input"); ax1.axis("off")
ax2.imshow(np.abs(feature_map), cmap="gray"); ax2.set_title("output of the vertical-edge kernel"); ax2.axis("off")
plt.show()

The output, called a **feature map**, lights up wherever the pattern occurs, at any position, with one set of 9 weights. Different kernels extract different structure; a small library of kernels already summarizes an image well. The two exercises build that intuition; then Part 2 hands the kernel design over to gradient descent.

### ✏️ Exercise 1
Design the kernel that detects **horizontal** edges, apply it to the same image, and display the result. (What relation does it have to `vertical_edge`?)

In [ ]:
# ✏️ horizontal_edge = ...



<details>
<summary>💡 Solution (click to open)</summary>

```python
horizontal_edge = vertical_edge.T      # transpose: top row minus bottom row

fm = convolve(image, horizontal_edge)
plt.figure(figsize=(5.5, 5.5))
plt.imshow(np.abs(fm), cmap="gray"); plt.axis("off")
plt.title("horizontal edges")
plt.show()
```

The transpose swaps rows and columns, so the detector responds to vertical brightness changes, meaning horizontal edges (the horizon, the camera's top edge). Kernels are directional pattern detectors.

</details>

### ✏️ Exercise 2
Not every kernel is an edge detector. Apply these two and describe each output in one sentence:

- the **averaging** kernel: a 3×3 grid of $1/9$
- the **sharpening** kernel: $\begin{pmatrix} 0 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 0 \end{pmatrix}$

In [ ]:
# ✏️ blur = ...
# ✏️ sharpen = ...



<details>
<summary>💡 Solution (click to open)</summary>

```python
blur = np.ones((3, 3)) / 9
sharpen = np.array([[0., -1., 0.], [-1., 5., -1.], [0., -1., 0.]])

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, (title, k) in zip(axes, [("input", None), ("blur", blur), ("sharpen", sharpen)]):
    ax.imshow(image if k is None else convolve(image, k).clip(0, 1), cmap="gray")
    ax.set_title(title); ax.axis("off")
plt.show()
```

Averaging replaces each pixel by its neighborhood mean, smoothing away fine detail (this is exactly the "blur" of photo editors). The sharpening kernel amplifies each pixel's difference from its neighbors, exaggerating detail. Every classic image filter is a convolution; the difference between filters is only the 9 numbers.

</details>

---
# Part 2: From kernels to layers

A CNN stacks three kinds of layers:

- **`nn.Conv2d(in, out, 3)`**: a convolution layer holding `out` learnable kernels, each spanning all `in` input channels; the output is one feature map per kernel. The kernels start random and are trained by backpropagation, like every weight since Session 7. Nobody designs them by hand.
- **`nn.ReLU()`**: the usual activation, applied elementwise to feature maps.
- **`nn.MaxPool2d(2)`**: replaces each 2×2 block by its maximum, halving width and height. This cuts downstream computation fourfold and makes the representation tolerant to shifts of a pixel or two, since the block's maximum survives them.

Stacking (conv, ReLU, pool) blocks produces a **feature hierarchy**: the first block's kernels see raw pixels and learn edge- and color-detectors; the next block sees pooled feature maps, so its kernels respond to *combinations* of edges over a wider area; and so on. This is the layered-abstraction idea of Session 7's slides, realized concretely.

The parameter economics are drastic and worth computing once by hand. A fully-connected layer from a 32×32×3 image to 256 units costs 3072 × 256 ≈ 786k weights. A `Conv2d(3, 16, 3)` layer costs 16 kernels × (3 × 3 × 3) weights + 16 biases = **448 parameters**, independent of the image size. The count, checked in code:

In [ ]:
import torch
from torch import nn

layer = nn.Conv2d(3, 16, kernel_size=3)
print("parameters of Conv2d(3, 16, 3):", sum(p.numel() for p in layer.parameters()))

### ✏️ Exercise 3
Compute by hand the parameter count of `nn.Conv2d(16, 32, kernel_size=3)` (16 input channels, 32 kernels of size 3×3), then verify with PyTorch.

In [ ]:
layer2 = nn.Conv2d(16, 32, kernel_size=3)
# ✏️ your hand computation first; then:
# print(sum(p.numel() for p in layer2.parameters()))



<details>
<summary>💡 Solution (click to open)</summary>

```python
print(32 * (16 * 3 * 3) + 32)                          # 4640
print(sum(p.numel() for p in layer2.parameters()))     # 4640
```

Each of the 32 kernels spans all 16 input channels: 16 × 3 × 3 = 144 weights plus one bias, and 32 × 145 = 4,640, regardless of image size. That independence lets one architecture scale from thumbnails to megapixel photos.

</details>

### ✏️ Exercise 4
Shape tracking, the everyday skill of CNN engineering. An input of shape `(3, 32, 32)` passes through:

`Conv2d(3, 16, 3, padding=1)` → `MaxPool2d(2)` → `Conv2d(16, 32, 3, padding=1)` → `MaxPool2d(2)` → `Flatten()`

Work out the shape after every step on paper (`padding=1` keeps width and height unchanged through a 3×3 convolution; pooling halves them). What length does `Flatten` produce? Verify by pushing a dummy tensor through the layers one at a time.

In [ ]:
x = torch.zeros(1, 3, 32, 32)
# ✏️ apply the layers step by step and print x.shape after each



<details>
<summary>💡 Solution (click to open)</summary>

```python
steps = [nn.Conv2d(3, 16, 3, padding=1), nn.MaxPool2d(2),
         nn.Conv2d(16, 32, 3, padding=1), nn.MaxPool2d(2), nn.Flatten()]
x = torch.zeros(1, 3, 32, 32)
for step in steps:
    x = step(x)
    print(step.__class__.__name__, tuple(x.shape))
```

`(16, 32, 32)` → `(16, 16, 16)` → `(32, 16, 16)` → `(32, 8, 8)` → `2048`. The 32 × 8 × 8 = 2,048 flattened length is where the first linear layer of Part 3's model gets its input size; mismatching this number is the single most common CNN bug.

</details>

---
# Part 3: A CNN on CIFAR-10, against the fully-connected network

The benchmark from Session 7: a fully-connected network with 789,258 parameters reached test accuracy **0.499** on CIFAR-10 (Adam, 10 epochs). Same dataset, same optimizer, same 10 epochs, convolutional architecture:

In [ ]:
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

train_data = torchvision.datasets.CIFAR10(root="data", train=True, download=True,
                                          transform=transforms.ToTensor())
test_data = torchvision.datasets.CIFAR10(root="data", train=False, download=True,
                                         transform=transforms.ToTensor())
print("train:", len(train_data), "| test:", len(test_data))

In [ ]:
torch.manual_seed(42)

cnn = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3 color channels → 16 feature maps
    nn.ReLU(),
    nn.MaxPool2d(2),                              # 32×32 → 16×16
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 maps → 32 maps
    nn.ReLU(),
    nn.MaxPool2d(2),                              # 16×16 → 8×8
    nn.Flatten(),
    nn.Linear(32 * 8 * 8, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

print("parameters:", sum(p.numel() for p in cnn.parameters()))

268,650 parameters, roughly a third of the fully-connected model. The training loop is unchanged from Session 7 Part 8; only `model` differs:

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

cnn = cnn.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)

for epoch in range(10):
    total = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(cnn(images), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(labels)
    print(f"epoch {epoch + 1:2d} | mean loss {total / len(train_data):.3f}")

In [ ]:
test_loader = DataLoader(test_data, batch_size=512)

def accuracy(model):
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            correct += (model(images.to(device)).argmax(dim=1).cpu() == labels).sum().item()
    return correct / len(test_data)

print("CNN test accuracy:", round(accuracy(cnn), 3))

The head-to-head result:

| | fully-connected (Session 7) | CNN (this part) |
|---|---|---|
| parameters | 789,258 | **268,650** |
| CIFAR-10 test accuracy | 0.499 | **≈ 0.69** |
| weights per pattern | one per absolute pixel position | one small kernel, shared everywhere |

Three times fewer parameters and twenty points more accuracy, from architecture alone: locality and weight sharing encode the structure of images that a fully-connected layer would have to discover from data. Matching the architecture to the structure of the data is the central design principle of deep learning; Session 10's transformers are the same principle applied to language.

### ✏️ Exercise 5
Compute the CNN's per-class accuracy and compare with the fully-connected network's numbers from Session 7: airplane 0.34, automobile 0.61, bird 0.34, cat 0.39, deer 0.37, dog 0.45, frog 0.64, horse 0.46, ship 0.72, truck 0.62. Which classes gained the most, and what do they have in common?

In [ ]:
# ✏️ per-class accuracy of the CNN



<details>
<summary>💡 Solution (click to open)</summary>

```python
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        all_preds.append(cnn(images.to(device)).argmax(dim=1).cpu())
        all_labels.append(labels)
preds = torch.cat(all_preds); labels = torch.cat(all_labels)

for c, name in enumerate(test_data.classes):
    mask = labels == c
    print(f"{name:12s} {(preds[mask] == c).float().mean():.2f}")
```

Every class improves; the largest jumps are airplane (0.34 → about 0.81) and bird (0.34 → about 0.60), classes whose subjects appear at highly variable positions and scales. Position-independent detectors help most exactly where position varied most.

</details>

### ✏️ Exercise 6
Weight sharing predicts that the CNN should tolerate small shifts of the input. Test the prediction: evaluate the trained CNN on test images shifted 2 and 4 pixels to the right (`torch.roll(images, shifts=k, dims=3)` inside the evaluation loop) and report the accuracies.

In [ ]:
# ✏️ shifted-input evaluation



<details>
<summary>💡 Solution (click to open)</summary>

```python
def accuracy_shifted(model, k):
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = torch.roll(images, shifts=k, dims=3)
            correct += (model(images.to(device)).argmax(dim=1).cpu() == labels).sum().item()
    return correct / len(test_data)

for k in (0, 2, 4):
    print(f"shift {k}px: {accuracy_shifted(cnn, k):.3f}")
```

Roughly 0.69 → 0.66 → 0.60: a graceful degradation (part of which is the artifact of `roll` wrapping pixels around the border). Shared kernels respond to the pattern wherever it lands, and pooling absorbs the residual misalignment. A fully-connected network, whose every weight is bound to one pixel position, has no such mechanism.

</details>

### ✏️ Exercise 7
Extend the model with a third block, `Conv2d(32, 64, 3, padding=1)` + ReLU + pooling, adjust the `Flatten`/`Linear` sizes with the Exercise 4 method, retrain, and report parameters and accuracy. One change at a time, measured, is the whole craft of architecture work.

In [ ]:
# ✏️ three-block CNN



<details>
<summary>💡 Solution (click to open)</summary>

```python
torch.manual_seed(42)
cnn3 = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # → 16×16×16
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # → 32×8×8
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # → 64×4×4
    nn.Flatten(),
    nn.Linear(64 * 4 * 4, 128), nn.ReLU(), nn.Linear(128, 10),
).to(device)
print("parameters:", sum(p.numel() for p in cnn3.parameters()))
# retrain with the same loop, then: print(round(accuracy(cnn3), 3))
```

The third block deepens the hierarchy while the extra pooling keeps the flattened size (64 × 4 × 4 = 1,024) *smaller* than before, so the model can end up with fewer parameters yet typically a few points more accuracy. Measure it; never assume an architecture change helped.

</details>

---
# Part 4: Inside the trained CNN

### 4.1 The learned kernels
The first layer's 16 kernels are 3×3×3, small enough to display as color patches (normalized for display):

In [ ]:
kernels = cnn[0].weight.detach().cpu()
kernels = (kernels - kernels.min()) / (kernels.max() - kernels.min())

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for j, ax in enumerate(axes.ravel()):
    ax.imshow(kernels[j].permute(1, 2, 0))
    ax.axis("off")
plt.suptitle("the 16 learned first-layer kernels (3×3, in color)")
plt.tight_layout(); plt.show()

At 3×3 resolution the informative view is what they *do*. Passing one test image through the first convolution and ReLU produces 16 feature maps:

In [ ]:
image, label = test_data[0]

with torch.no_grad():
    maps = nn.Sequential(cnn[0], cnn[1])(image.unsqueeze(0).to(device))[0].cpu()

fig, axes = plt.subplots(2, 9, figsize=(13, 3.2))
axes[0, 0].imshow(image.permute(1, 2, 0)); axes[0, 0].set_title(test_data.classes[label], fontsize=8)
axes[1, 0].axis("off")
for j in range(8):
    axes[0, j + 1].imshow(maps[j], cmap="gray")
    axes[1, j + 1].imshow(maps[j + 8], cmap="gray")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("input and its 16 first-layer feature maps")
plt.tight_layout(); plt.show()

Different maps respond to different structure: oriented edges, color regions. These are the same kind of detectors as Part 1's hand-designed kernels, except the network **chose them** by gradient descent because they help classification. Contrast Session 7's dense-layer weight images, which showed no readable structure: given a spatial vocabulary, interpretable detectors emerge even in a small network.

### ✏️ Exercise 8
Display the feature maps after the second convolution block (layers `cnn[0]` through `cnn[4]`); they are 16×16 and there are 32, show the first 16. How do they differ in character from the first-layer maps?

In [ ]:
# ✏️ second-block feature maps



<details>
<summary>💡 Solution (click to open)</summary>

```python
with torch.no_grad():
    maps2 = nn.Sequential(*cnn[:5])(image.unsqueeze(0).to(device))[0].cpu()
print(maps2.shape)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for j, ax in enumerate(axes.ravel()):
    ax.imshow(maps2[j], cmap="gray")
    ax.axis("off")
plt.suptitle("16 of the 32 second-layer feature maps (16×16)")
plt.tight_layout(); plt.show()
```

Coarser and more abstract: each value summarizes a larger patch of the original image, and the maps respond to combinations of first-layer patterns rather than raw edges. The hierarchy, observed directly.

</details>

### ✏️ Exercise 9
For the same test image, find which first-layer feature map is the most strongly activated on average (`maps[j].mean()`), display that map next to the input, and describe what its kernel appears to detect.

In [ ]:
# ✏️ strongest feature map



<details>
<summary>💡 Solution (click to open)</summary>

```python
j = maps.mean(dim=(1, 2)).argmax().item()
print("strongest map:", j)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3.5))
ax1.imshow(image.permute(1, 2, 0)); ax1.set_title("input"); ax1.axis("off")
ax2.imshow(maps[j], cmap="gray"); ax2.set_title(f"feature map {j}"); ax2.axis("off")
plt.show()
```

Which map wins depends on the image content (for many images it is a map responding to a dominant color or a strong contour). Reading a model's internal activations against its input, exactly as here, is the basic tool of CNN debugging and interpretability work.

</details>

---
# Part 5: Data augmentation

A model should not care whether the cat faces left or right, sits slightly off-center, or was photographed a bit darker. **Data augmentation** builds that into training: every time an image is used, a random label-preserving transform (flip, small shift or rotation, brightness change) is applied first, so the network sees a fresh variant in every epoch and cannot memorize exact pixels. It is a standard part of real training pipelines and one practical answer to overfitting (Session 3).

The standard library is **albumentations** (preinstalled on Colab; locally, install once with `pip install albumentations`). A pipeline is a list of transforms, each with its own probability; calling it on an image applies a random draw:

In [ ]:
import albumentations as A

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.9, 1.1), rotate=(-15, 15), p=1.0),
    A.RandomBrightnessContrast(p=0.5),
])

cat = (load_image("chelsea.png") * 255).astype(np.uint8)   # albumentations expects uint8

fig, axes = plt.subplots(2, 4, figsize=(13, 5.5))
axes[0, 0].imshow(cat); axes[0, 0].set_title("original", fontsize=9)
for ax in axes.ravel()[1:]:
    ax.imshow(augment(image=cat)["image"])
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("one photo, seven random augmentations")
plt.tight_layout(); plt.show()

Same cat, same label, different pixels on every draw. The same pipeline on a CIFAR-10 sample, at the resolution the network actually sees:

In [ ]:
sample = train_data.data[7]                  # numpy, 32x32x3, uint8

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
axes[0].imshow(sample); axes[0].set_title("original", fontsize=8)
for ax in axes[1:]:
    ax.imshow(augment(image=sample)["image"])
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

Wiring augmentation into training takes one step: apply the pipeline inside the dataset, so every batch is drawn fresh:

In [ ]:
class AugmentedCIFAR(torch.utils.data.Dataset):
    def __init__(self, base, pipeline):
        self.base = base
        self.pipeline = pipeline
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        img = self.pipeline(image=self.base.data[i])["image"]
        return torch.tensor(img).permute(2, 0, 1).float() / 255.0, self.base.targets[i]

augmented_loader = DataLoader(AugmentedCIFAR(train_data, augment), batch_size=128, shuffle=True)

images, labels = next(iter(augmented_loader))
print("one batch, freshly augmented:", tuple(images.shape))

Passing `augmented_loader` to the Part 3 training loop is all it takes. The payoff appears when a model starts to overfit: the network can never see the exact same picture twice, so memorization stops being an option.

### ✏️ Exercise 10
Add `A.VerticalFlip(p=1.0)` to a copy of the pipeline and display a few CIFAR samples. The transform is technically valid; why is it a poor choice for this dataset? (What would an upside-down truck teach the network?)

In [ ]:
# ✏️ vertical-flip pipeline and a look at its output



<details>
<summary>💡 Solution (click to open)</summary>

```python
flipped = A.Compose([A.VerticalFlip(p=1.0)])

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, i in zip(axes, range(6)):
    ax.imshow(flipped(image=train_data.data[i])["image"])
    ax.axis("off")
plt.show()
```

Upside-down trucks, deer, and ships essentially never occur in real photographs, so the network would spend capacity on inputs it will never meet at test time. Augmentation must stay inside the real data distribution; choosing the transforms is domain knowledge. (For satellite or microscope images, where every rotation is natural, rotations and flips of all kinds are exactly right.)

</details>

### ✏️ Exercise 11
Build a pipeline of your own with two transforms not used above, for example `A.GaussianBlur` and `A.CoarseDropout`, and display eight variants of the cat photo. The albumentations documentation lists dozens of options.

In [ ]:
# ✏️ your pipeline, eight variants



<details>
<summary>💡 Solution (click to open)</summary>

```python
mine = A.Compose([
    A.GaussianBlur(p=0.5),
    A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(20, 60),
                    hole_width_range=(20, 60), p=1.0),
])

fig, axes = plt.subplots(2, 4, figsize=(13, 5.5))
for ax in axes.ravel():
    ax.imshow(mine(image=cat)["image"])
    ax.axis("off")
plt.show()
```

`CoarseDropout` blanks random rectangles, forcing the network to use the whole object instead of one telltale patch; blur simulates focus variation. Both are common in production pipelines.

</details>

---
# Part 6: Transfer learning with ResNet18

Scaling Part 3's recipe (more blocks, more data, more compute) is how vision was solved industrially. **ResNet18** is a standard 18-layer CNN with 11.7 million parameters, trained on ImageNet (1.2 million photos, 1,000 categories); `torchvision` ships the trained weights (~45 MB, downloaded once). Out of the box it classifies photographs:

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
resnet = resnet18(weights=weights).eval()
preprocess = weights.transforms()          # the resizing/normalization ResNet expects
categories = weights.meta["categories"]

for name, img in [("chelsea", load_image("chelsea.png")), ("coffee", load_image("coffee.png"))]:
    x = preprocess(torch.tensor(img).permute(2, 0, 1)).unsqueeze(0)
    with torch.no_grad():
        probs = resnet(x).softmax(dim=1)[0]
    top = probs.topk(3)
    print(name, "→", [(categories[i], round(v.item(), 2)) for v, i in zip(top.values, top.indices)])

### 6.1 The professional workflow: reuse the features
Almost nobody trains vision models from scratch anymore. The standard move, **transfer learning**, treats the pretrained network's convolutional stack as a general-purpose feature extractor and trains only a new classifier head on one's own data.

The experiment below makes the case quantitatively. Replace ResNet18's final layer with the identity, so the network outputs its 512-dimensional feature vector; run **5,000** CIFAR-10 training images (a tenth of the dataset) through it; and train a logistic regression on those features. No convolutional weight is updated. (The two extraction cells are the slow ones on CPU, about 2 minutes total; evaluation uses 2,000 test images.)

In [ ]:
from torch.utils.data import Subset

backbone = resnet18(weights=weights)
backbone.fc = nn.Identity()               # drop the 1000-class head: output = 512-dim features
backbone.eval()

def extract_features(dataset, n):
    features, labels = [], []
    with torch.no_grad():
        for images, ys in DataLoader(Subset(dataset, range(n)), batch_size=64):
            features.append(backbone(preprocess(images)))
            labels.append(ys)
    return torch.cat(features).numpy(), torch.cat(labels).numpy()

F_train, y_train = extract_features(train_data, 5000)
F_test, y_test = extract_features(test_data, 2000)
print("feature matrices:", F_train.shape, F_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

probe = LogisticRegression(max_iter=2000)
probe.fit(F_train, y_train)

print("transfer accuracy (5,000 training images):", round(probe.score(F_test, y_test), 3))

The scoreboard for this session so far:

| model | training images | trained parameters | CIFAR-10 accuracy |
|---|---|---|---|
| fully-connected (Session 7) | 50,000 | 789k | 0.499 |
| CNN from scratch (Part 3) | 50,000 | 269k | ≈ 0.69 |
| **ResNet18 features + linear classifier** | **5,000** | **5k (the classifier only)** | **≈ 0.82** |

A linear model on pretrained features, using a **tenth of the data** and updating no convolutional weight, beats the CNN we trained on everything. The features ResNet learned from 1.2 million photos transfer to a dataset it never saw; that is why transfer learning is the default in professional practice, and why "how much labeled data do you have" is no longer the first blocker of applied vision.

### 6.2 Real fine-tuning
The probe kept the backbone frozen. **Fine-tuning** goes further: replace the head with a fresh 10-class layer and update *all* weights on the new data, with a small learning rate so the pretrained features are adjusted rather than destroyed. On a Colab T4 GPU the two epochs below take about a minute; on CPU expect roughly ten:

In [ ]:
torch.manual_seed(42)
finetuned = resnet18(weights=weights)
finetuned.fc = nn.Linear(512, 10)                 # new head for our 10 classes
finetuned = finetuned.to(device)

optimizer = torch.optim.Adam(finetuned.parameters(), lr=1e-4)   # small: adjust, do not destroy
train_5k = DataLoader(Subset(train_data, range(5000)), batch_size=32, shuffle=True)

for epoch in range(2):
    finetuned.train()
    total = 0.0
    for images, labels in train_5k:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(finetuned(preprocess(images)), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(labels)
    print(f"epoch {epoch + 1} | mean loss {total / 5000:.3f}")

finetuned.eval()
correct = 0
with torch.no_grad():
    for images, labels in DataLoader(Subset(test_data, range(2000)), batch_size=64):
        images = images.to(device)
        correct += (finetuned(preprocess(images)).argmax(dim=1).cpu() == labels).sum().item()
print("fine-tuned accuracy:", round(correct / 2000, 3))

The scoreboard, complete:

| model | training images | trained parameters | CIFAR-10 accuracy |
|---|---|---|---|
| fully-connected (Session 7) | 50,000 | 789k | 0.499 |
| CNN from scratch (Part 3) | 50,000 | 269k | ≈ 0.69 |
| ResNet18 features + linear classifier | 5,000 | 5k | ≈ 0.82 |
| **fine-tuned ResNet18** | **5,000** | **11.7M** | **≈ 0.87** |

Adapting the pretrained features to the task beats freezing them, at the cost of more computation. This pair of options, frozen features for speed, fine-tuning for accuracy, is the standard menu of applied computer vision, and both ran here on a tenth of the dataset.

### ✏️ Exercise 12
Data efficiency: refit the logistic regression on only the first **1,000** training rows of `F_train` and evaluate. Compare all four numbers (FC, scratch CNN, 5,000-image probe, 1,000-image probe). Where does labeled data stop being the bottleneck?

In [ ]:
# ✏️ 1,000-image probe



<details>
<summary>💡 Solution (click to open)</summary>

```python
probe_small = LogisticRegression(max_iter=2000)
probe_small.fit(F_train[:1000], y_train[:1000])
print(round(probe_small.score(F_test, y_test), 3))
```

About 0.79: with **one fiftieth** of the labels, the pretrained-features pipeline still clearly beats both networks trained from scratch on the full 50,000 images. Pretraining shifted the bottleneck from labeled data to feature quality, which is precisely the shift that later made large language models practical (Session 10).

</details>

### ✏️ Exercise 13
Classify `load_image("astronaut.png")` (a portrait of astronaut Eileen Collins) with the ResNet classifier, printing the top 5. The result will look broken. Inspect `categories` and state precisely what is missing. Keep your conclusion; Part 7 returns to this exact image.

In [ ]:
# ✏️ astronaut, top 5



<details>
<summary>💡 Solution (click to open)</summary>

```python
img = load_image("astronaut.png")
x = preprocess(torch.tensor(img).permute(2, 0, 1)).unsqueeze(0)
with torch.no_grad():
    probs = resnet(x).softmax(dim=1)[0]
top = probs.topk(5)
for value, idx in zip(top.values, top.indices):
    print(f"{categories[idx]:20s} {value.item():.2f}")

print("person" in categories)      # False
```

Low-confidence guesses about background objects, because **ImageNet's 1,000 categories contain no "person" class**. A classifier answers only from its label set; out-of-vocabulary inputs are silently mapped onto whatever is available. Knowing a model's output space matters as much as its accuracy.

</details>

---
# Part 7: Professional tools: detection and segmentation

Classification answers one question per image: *what is this?* Production systems usually need more:

- **Object detection**: *what is where?* The model outputs a list of bounding boxes, each with a class and a confidence score. This is the technology of plate readers, pedestrian detection, and shelf-scanning robots.
- **Semantic segmentation**: *which pixels belong to what?* The model classifies every pixel, producing a mask. This is the technology of medical image analysis and autonomous-driving scene understanding.

Both are CNNs (with detection- or segmentation-specific heads on a backbone like ResNet), both pretrained and shipped by `torchvision`. **Faster R-CNN**, trained on the COCO dataset (91 everyday categories, person included), first (~167 MB download):

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

det_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn(weights=det_weights).eval()
det_categories = det_weights.meta["categories"]
print("COCO categories:", len(det_categories), "| person included:", "person" in det_categories)

In [ ]:
import matplotlib.patches as patches

def show_detections(img, threshold=0.7):
    x = torch.tensor(img).permute(2, 0, 1).float()
    with torch.no_grad():
        det = detector([x])[0]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img); ax.axis("off")
    for box, label, score in zip(det["boxes"], det["labels"], det["scores"]):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, color="red", linewidth=2))
        ax.text(x1, y1 - 6, f"{det_categories[label]} {score:.2f}",
                color="white", fontsize=9, bbox=dict(facecolor="red", alpha=0.8))
    plt.show()

show_detections(load_image("astronaut.png"))
show_detections(load_image("coffee.png"))

Two results worth reading closely. The astronaut image, which the ImageNet *classifier* could only mislabel (Exercise 13), is handled correctly here: **person, confidence 1.00, localized with a box**, because the COCO label set contains the class and the detection task predicts *where*, not just *what*. And in the coffee photo the detector separates the scene into its parts: the table, the cup, the spoons, each with its own box and score. Same convolutional machinery, different head, different label set; choosing the right task formulation and training data is as decisive as the architecture.

### ✏️ Exercise 14
Rerun `show_detections(load_image("coffee.png"))` with `threshold=0.5` and with `threshold=0.9`, and count the boxes each time. What does the threshold trade off, and which setting would a supermarket shelf-scanner want versus a medical screening tool?

In [ ]:
# ✏️ two thresholds



<details>
<summary>💡 Solution (click to open)</summary>

```python
show_detections(load_image("coffee.png"), threshold=0.5)
show_detections(load_image("coffee.png"), threshold=0.9)
```

Low threshold: more boxes, including duplicates and low-confidence guesses (higher recall, lower precision). High threshold: only the confident boxes survive (higher precision, lower recall). The Session 9 spam lesson generalizes: the operating point depends on the cost of each error type. A screening tool that must not miss anything runs at a low threshold and accepts false alarms; a fully automated pipeline runs high.

</details>

### ✏️ Exercise 15
Semantic segmentation with **DeepLabV3** (`torchvision.models.segmentation.deeplabv3_resnet50`, ~160 MB). Load it with its default weights, run `load_image("astronaut.png")` through it, take the per-pixel `argmax` of the output, and display the mask beside the image. Which classes appear in the mask (`seg_weights.meta["categories"]`)?

In [ ]:
# ✏️ segmentation



<details>
<summary>💡 Solution (click to open)</summary>

```python
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

seg_weights = DeepLabV3_ResNet50_Weights.DEFAULT
segmenter = deeplabv3_resnet50(weights=seg_weights).eval()
seg_pre = seg_weights.transforms()

img = load_image("astronaut.png")
x = seg_pre(torch.tensor(img).permute(2, 0, 1)).unsqueeze(0)
with torch.no_grad():
    out = segmenter(x)["out"][0]
mask = out.argmax(dim=0)

print("classes in mask:", [seg_weights.meta["categories"][i] for i in mask.unique()])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(img); ax1.set_title("input"); ax1.axis("off")
ax2.imshow(mask); ax2.set_title("per-pixel class"); ax2.axis("off")
plt.show()
```

The mask separates `person` pixels from background: a decision for every pixel rather than one per image or per box. Classification, detection, and segmentation form the standard ladder of vision tasks; all three ran here on the same laptop with pretrained weights.

</details>

---
# Part 8: Summary and next steps

The session's results in one table:

| task | model | data used | result |
|---|---|---|---|
| classification | fully-connected (S7) | 50k images | 0.499 |
| classification | CNN from scratch | 50k images | ≈ 0.69 |
| classification | ResNet18 features + linear | 5k images | ≈ 0.82 |
| classification | fine-tuned ResNet18 | 5k images | ≈ 0.87 |
| detection | Faster R-CNN (pretrained) | 0 of ours | person 1.00, boxed |
| segmentation | DeepLabV3 (pretrained) | 0 of ours | per-pixel mask |

The argument, start to finish: convolution is a local, weight-sharing weighted sum (implemented in numpy); layers of learnable kernels build a feature hierarchy (trained and inspected); the architecture beats a fully-connected network with a third of the parameters; augmentation extends the data with label-preserving variation; pretrained backbones make features a commodity, so a linear classifier on a tenth of the data wins and a brief fine-tune wins further; and with the right heads, the same backbones solve detection and segmentation at production quality.

### ✏️ Exercise 16: homework
1. Take the three-block CNN of Exercise 7 past **0.75** test accuracy (longer training, more kernels, both). Log every change and its measured effect.
2. Kaggle's [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer): replace your Session 7 `MLPClassifier` submission with a small CNN and compare leaderboard scores.
3. On a Colab GPU, rerun the Part 6 fine-tuning with the Part 5 augmentation pipeline inside the training loader and 3 or more epochs. Measure whether you can pass 0.90.

*Next session: the same journey for text, from words as numbers to models that read.*